# 第12讲 · 机器学习与 KNN（含决策树）—— 鸢尾花分类

本 notebook 对应课件 `lecture12.md` 与脚本 `knn_classifier.py`。

**一句话目标**：给定鸢尾花 4 维特征（花萼长/宽、花瓣长/宽），判断它属于三种中的哪一类（setosa / versicolor / virginica）。

完整流程：
1. 加载数据 → 划分训练/测试
2. 训练决策树与 KNN
3. 评估准确率
4. K 值实验
5. 维度转换对比（4 维 vs 2 维）
6. 用 NumPy 手写最小 KNN

依赖：`numpy`、`scikit-learn`、`matplotlib`（已在 `.venv` 中）。

In [ ]:
# 让绘图直接嵌入 notebook（不弹窗）
%matplotlib inline

import numpy as np
import matplotlib.pyplot as plt
from matplotlib.colors import ListedColormap

from plots import setup_zh_font

from sklearn.datasets import load_iris
from sklearn.model_selection import train_test_split
from sklearn.tree import DecisionTreeClassifier
from sklearn.neighbors import KNeighborsClassifier
from sklearn.metrics import accuracy_score

# 中文字体设置，避免图标/坐标轴中文显示为方框
setup_zh_font()

## 1. 加载数据 + 训练/测试划分

`load_iris()` 返回 150 个样本 × 4 个特征，共 3 类。`train_test_split` 按 **分层抽样**（`stratify=y`，保证每类在训练/测试集中比例一致）切出 30% 测试集。固定 `random_state=42` 保证结果可复现。

In [ ]:
iris = load_iris()
X, y = iris.data, iris.target          # 150 样本 × 4 特征，3 类
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.3, random_state=42, stratify=y)
print(f"训练集形状: {X_train.shape}   测试集形状: {X_test.shape}")

## 2. 训练决策树 与 KNN

- **决策树** `DecisionTreeClassifier(max_depth=3)`：限制深度 3，防止过拟合。
- **KNN** `KNeighborsClassifier(n_neighbors=5)`：取最近 5 个训练样本做多数表决。

两者都用 `fit(X_train, y_train)` 训练，再用 `predict(X_test)` 预测，`accuracy_score` 算准确率。

In [ ]:
dt = DecisionTreeClassifier(max_depth=3, random_state=42)
dt.fit(X_train, y_train)

knn = KNeighborsClassifier(n_neighbors=5)
knn.fit(X_train, y_train)

y_pred_dt = dt.predict(X_test)
y_pred_knn = knn.predict(X_test)
print(f"决策树准确率: {accuracy_score(y_test, y_pred_dt):.3f}")
print(f"KNN(k=5) 准确率: {accuracy_score(y_test, y_pred_knn):.3f}")

## 3. K 值的影响（重点实验）

Iris 本身分类难度低，准确率差异很小；**K 值差异在含噪声的数据上才明显**。

- `K` 太小（如 1）：对噪声敏感、决策边界碎。
- `K` 太大：过度平均，可能淹没真实类别规律。

In [ ]:
print("[k 值实验] 不同 K 的准确率（Iris 本身不难，差异主要体现在噪声数据）")
for k in [1, 3, 5, 10, 20]:
    m = KNeighborsClassifier(n_neighbors=k).fit(X_train, y_train)
    print(f"  k={k:<3} 准确率={accuracy_score(y_test, m.predict(X_test)):.3f}")

## 4. 维度转换（4 维 vs 2 维）

4 维原始特征无法直接画散点图。这里降到 2 维（只用**花瓣长、花瓣宽**，即第 2、3 列）便于可视化与解释。

> 4 维信息更多，精度通常更高；2 维好画图、好解释。下一格画 KNN 的 2D 决策边界。

In [ ]:
# 4 维全特征 -> 2 维花瓣特征
X2 = X[:, 2:]
print(f"4 维全特征 shape={X.shape} → 2 维花瓣特征 shape={X2.shape}")
print("  4 维信息更多、精度通常更高；2 维好画散点图、好解释（见下图）")

### 2D 决策边界可视化

在 2D 花瓣特征空间用 `KNeighborsClassifier(k=5)` 铺满网格预测类别，画 `contourf` 色块边界，再叠上原始样本点。

In [ ]:
cmap_light = ListedColormap(["#ffd6cf", "#cfeaff", "#e2f6d5"])
cmap_bold = ListedColormap(["#d95763", "#3a7ca5", "#4a8f54"])

knn2 = KNeighborsClassifier(n_neighbors=5).fit(X2, y)
x_min, x_max = X2[:, 0].min() - 0.5, X2[:, 0].max() + 0.5
y_min, y_max = X2[:, 1].min() - 0.5, X2[:, 1].max() + 0.5
xx, yy = np.meshgrid(np.linspace(x_min, x_max, 300),
                     np.linspace(y_min, y_max, 300))
Z = knn2.predict(np.c_[xx.ravel(), yy.ravel()]).reshape(xx.shape)

plt.figure(figsize=(6.4, 5.2))
plt.contourf(xx, yy, Z, cmap=cmap_light, alpha=0.8)
plt.scatter(X2[:, 0], X2[:, 1], c=y, cmap=cmap_bold,
            edgecolor="k", s=40)
plt.xlabel("花瓣长度 (Petal length)")
plt.ylabel("花瓣宽度 (Petal width)")
plt.title(f"KNN(k=5) 在 2 维花瓣特征上的决策边界")
plt.show()

## 5. 用 NumPy 手写最小 KNN（不调库）

三要素：**算距离 → 找 K 邻居 → 多数表决**。

- 距离用欧氏距离平方：`np.sum((X_train - x) ** 2, axis=1)`。
- 找最近 `k` 个：`np.argsort(d2)[:k]`。
- 多数表决：`np.unique(..., return_counts=True)`，取出现最多标签。

下面用 2 维数据与 `sklearn` 的实现对比，验证手写版正确性。

In [ ]:
def knn_predict(X_train, y_train, X_test, k=5):
    """对 X_test 每个样本，用欧氏距离找最近的 k 个训练样本，多数表决定类别。"""
    preds = []
    for x in X_test:
        d2 = np.sum((X_train - x) ** 2, axis=1)   # 欧氏距离平方
        idx = np.argsort(d2)[:k]                   # 距离最小的 k 个邻居的索引
        labels, counts = np.unique(y_train[idx], return_counts=True)
        preds.append(labels[np.argmax(counts)])
    return np.asarray(preds)

In [ ]:
X_tr2, X_te2, y_tr2, y_te2 = train_test_split(
    X2, y, test_size=0.3, random_state=42, stratify=y)
hand_pred = knn_predict(X_tr2, y_tr2, X_te2, k=5)
sk_pred = KNeighborsClassifier(n_neighbors=5).fit(X_tr2, y_tr2).predict(X_te2)

print(f"[手写 KNN vs sklearn] 手写准确率={accuracy_score(y_te2, hand_pred):.3f}"
      f"   sklearn准确率={accuracy_score(y_te2, sk_pred):.3f}")

## 小结

- 决策树与 KNN 都能在 Iris 上取得很高准确率。
- K 值是个超参数：过大过小都影响泛化，需要调。
- 高维特征不好可视化，常降维到 2 维画决策边界。
- 手写 KNN 验证了算法本质：距离 + 邻域 + 投票。